# Comparing Parametric and Nonparametric Forecasting Models 

Monte carlo simulation study comparing OLS, kernel regression, and polynomial series estimators on out-of-sample forecasting performance, plus a bandwidth-selection comparison for kernel density estimation.

**Tools:** NumPy, pandas, statsmodels, scikit-learn, Matplotlib


***
##  Part 1 — Data Generating Process

The simulation study is built around the following data generating process (DGP):

$$y_i = 0.1 + 0.5x_{1i} - 0.5\ln(x_{2i}) + u_i \tag{1}$$

where:
- $u_i \sim N(0,1)$
- $x_{1i} \sim \text{Uniform}(0.5, 2.5)$
- $x_{2i} \sim \text{Beta}(0.5, 0.7)$

The function below, `dgp`, simulates this process and returns a Pandas DataFrame containing $y$, $x_1$, and $x_2$.

***


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.optimize import minimize_scalar
from scipy import stats
from itertools import product
import random
from scipy.stats import beta as beta_dist
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from itertools import product
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_squared_error


SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# Create DGP using the true model
def dgp(n=1000):
    u = np.random.normal(loc=0, scale=1, size=n) # draws u~N(0,1) 
    x1 = np.random.uniform(low=0.5, high=2.5, size=n) #draws x1~(0.5,2.5)
    x2 = np.random.beta(a=0.5, b=0.7, size=n) # draws x2~Beta(0.5,0.7)
    y = 0.1 + 0.5 * x1 - 0.5 * np.log(x2) + u # generates y using the true model 
    
    return pd.DataFrame({'y': y, 'x1': x1, 'x2': x2}) # returns all as a DataFrame

print(dgp(n=5)) # check with 5 observations

***
## Part 2 — Kernel Density Estimation: Plug-in vs. Cross-Validated Bandwidths

Using a sample of $n = 10{,}000$ drawn from the DGP above (`sim_data`), this section estimates and compares kernel density estimates for $x_1$ and $x_2$ using the Epanechnikov kernel under two bandwidth-selection rules:

1. The plug-in (Silverman) rule of thumb: $h_{opt,E} = 1.06 \cdot n^{-1/5} \cdot \min\{\hat{\sigma}, IQR/1.34\}$
2. Least-squares cross-validation (LSCV)

The goal is to assess how each bandwidth rule performs depending on how closely the true distribution resembles a Gaussian — testing this against both a roughly-normal distribution ($x_1$) and a sharply non-normal, U-shaped distribution ($x_2$).

***


Using the Epanechnikov kernel the kernel density estimates for $x_1$ and $x_2$ are estimated:

$$K(u) = \frac{3}{4}(1 - u^2)\mathbf{1}(|u| \leq 1)$$

The Epanechnikov kernel assigns zero weight to observations outside the bandwidth window, so only nearby $x$'s contribute to each estimate. The kernel density estimate is evaluated over a grid of 300 evenly spaced $x$ values, where each element of the distance matrix is scaled by bandwidth $h$:

$$\hat{f}(x) = \frac{1}{nh} \sum_{i=1}^{n} K\left(\frac{x - x_i}{h}\right)$$

Two methods are used to choose the bandwidth; firstly, the plug-in (Silverman) rule:

$$h_{opt} = 1.06 \cdot n^{-1/5} \cdot \min\left(\hat{\sigma},\ \frac{\widehat{IQR}}{1.34}\right)$$

which chooses $h$ by minimising the asymptotic Mean Integrated Squared Error (MISE)under the assumption that the true density is Gaussian. 

Second, least-squares cross-validation (LSCV), which chooses $h$ by minimising:

$$CV(h) = \int \hat{f}(x)^2\, dx - \frac{2}{n} \sum_{i=1}^{n} \hat{f}_{-i}(x_i)$$

where $\hat{f}_{-i}(x_i)$ is the leave-one-out density estimate at $x_i$. LSCV chooses $h$ by minimising an estimate of the Mean Squared Error (MSE) directly from the data, without any parametric assumptions.

For $x_1 ~ U(0.5, 2.5)$, both bandwidths are expected to be similar since a uniform distribution is close to the normal distribution and so the plug-in rule's normality assumption is not severely violated. For $x_2 ~ Beta(0.5, 0.7)$, the two bandwidths are expected to be very different. The plug-in rule will likely oversmooth, ignoring the sharp spikes near 0 and 1 that characterise the U-shaped distribution. LSCV on the other hand, should choose a much smaller bandwidth that captures these boundary peaks. In general, LSCV is expected to be more true to the actual distributional shape. This has the cost of potentially overfitting when the true distribution is simpler, like $x_1$, as it's bandwidth choice will be too small, and will therefore track sample noise rather than trends in the distribution.

In [ ]:
np.random.seed(SEED)

# 2.1-----------------------------
# Generate sample of 10,000 observations using dgp
sim_data = dgp(n=10000)

# 2.2 -----------------------------
# Epanechnikov KDE using plug-in bandwidth

# Epanechinkov set up
def K_epa(u):   #defines Epanechinkov kernel function
    return np.where(np.abs(u) <= 1, 0.75 * (1.0 - u**2), 0.0)   # returns 0.75*(1-u²) where |u|<=1, else 0

def kde_epa(x_eval, x_data, h): #computes kernel density estimate
                                #x_eval: points to est density function at
                                #x_data: observed data
                                #h: bandwidth
    x_eval = np.asarray(x_eval)
    x_data = np.asarray(x_data) # checks inputs are npy arrays
    U = (x_eval[:, np.newaxis] - x_data[np.newaxis, :]) / h  #builds an (n_eval × n_data) matrix of scaled distances
                                                             #x_eval[:,np.newaxis] reshapes to column vector
                                                             #each row i contains (x_eval[i] - x_data[j]) / h for all j                                  
    return K_epa(U).mean(axis=1) / h #applies kernel to every element of U
                                     #averages kernel weights across data points (axis=1),
                                     #then divides by h to normalise so the result integrates to 1

# Plug-in bandwidth
def plug_in_bw(x):  #defines Silverman's rule of thumb bandwidth
    x = np.asarray(x) #ensures input is a numpy array
    n = len(x)   #stores number of observations
    sigma_hat = np.std(x, ddof=1)   #sample standard deviation (ddof=1 dvides by n-1, unbiased estimator)
    iqr_hat = np.percentile(x, 75) - np.percentile(x, 25)   #IQR (75th - 25th) — more robust to outliers and heavy tails than std alone

    return 1.06 * n**(-0.2) * min(sigma_hat, iqr_hat / 1.34)  #Silverman bandwidth formula


h_x1_pi = plug_in_bw(sim_data['x1']) #optimal plug-in bandwidths for x1
h_x2_pi = plug_in_bw(sim_data['x2']) #optimal plug-in bandwidths for x2

# 2.3 ---------------------------
# Cross-validated bandwidth (LSCV)
def lscv(h, x_data): # computes LSCV score
    n = len(x_data)
    U = (x_data[:, np.newaxis] - x_data[np.newaxis, :]) / h #creates nxn matrix of pairwise scaled distances
    K = K_epa(U) #applies the Epanechnikov kernel to every element, giving kernel weight for all pairs

    # Approximates desnity squared integral 
    x_grid = np.linspace(x_data.min() - 3*h, x_data.max() + 3*h, 300) #creates 300 evenly spaced points
                                                                       #extends 3 bandwidths (on each side) beyond the data range to capture tails
    f_hat = kde_epa(x_grid, x_data, h)   #evaluates KDE at every grid point using current bandwidth h
    integral = np.trapezoid(f_hat**2, x_grid)  #integrates f_hat^2 over the grid using the trapezoid rule
                                           #approximates the first term of the LSCV objective

    # Leave-one-out KDE at each data point
    np.fill_diagonal(K, 0.0)   #sets diagonal of kernel matrix to 0 (removes oberservations self influence)
    loo_dens = K.sum(axis=1) / ((n - 1) * h)   #finds leave-one-out KDE at each x_i
                                               #for each point i, sums kernel weights from all other points
                                               #normalises by (n-1)*h
    # Return CV criteria to minimise
    return integral - 2.0 * np.mean(loo_dens)  #minimising this balances fit against over-smoothing
                                               #np.mean(loo_dens) rewards the density for assinging high probability to observed data points

def cv_bandwidth(x, seed=0):
    x  = np.asarray(x, dtype=float) #converts input to float numpy array
    h_pi = plug_in_bw(x)    #uses the plug-in bandwidth to set the search range
    res  = minimize_scalar(lscv, args=(x,),
                           bounds=(h_pi * 0.01, h_pi * 5.0),
                           method='bounded')    #bounded search for h minimising CV
    
    return res.x     #gives optimal bandwidth found

h_x1_cv = cv_bandwidth(sim_data['x1'].values) #runs CV bandwith selection, converts to numpy array
h_x2_cv = cv_bandwidth(sim_data['x2'].values)

In [ ]:
# Results table ----------------------------------
bw_table = pd.DataFrame({'Variable': ['x1', 'x2'],
                         'Plug-in Bandwidth': [h_x1_pi, h_x2_pi],
                         'CV Bandwidth': [h_x1_cv, h_x2_cv],
                         }).set_index('Variable')   #makes 'variable' column the row index
bw_table = bw_table.round(5)    #round output to 5dpt

print("\nTable 1: Bandwidth Choice Results")
print("=" * 50)
print(bw_table.to_string())     #prints table w/o truncation
print("=" * 50)


# Plots ----------------------------------
# Create grids for plotting
x1_grid = np.linspace(sim_data['x1'].min() - 0.05,          #creates x-axis points (where KDE is evaluated) for plotting
                      sim_data['x1'].max() + 0.05, 300)     #300 points just beyond x_1's data range (+/-0.05)
                                            
x2_grid = np.linspace(0.001, 0.999, 300)    #creates 300 points between 0.001-0.999 for x_2
                                            #fixed bounds as ~Beta(0,1)
                                            #avoids 0 and 1 as beta density may be extreme at the boundaries

# Compute KDEs
f_x1_pi = kde_epa(x1_grid, sim_data['x1'].values, h_x1_pi)  #every grid point using plug-in bandwidth for x_1
f_x2_pi = kde_epa(x2_grid, sim_data['x2'].values, h_x2_pi)
f_x1_cv = kde_epa(x1_grid, sim_data['x1'].values, h_x1_cv)  #every grid point using CV bandwidth for x_1
f_x2_cv = kde_epa(x2_grid, sim_data['x2'].values, h_x2_cv)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))    #axes: creates 2x2 grid of subplots
                                                    #individual plots: axes[row, col]

# Figure 1: x1 Plug-in      
axes[0, 0].plot(x1_grid, f_x1_pi, 
                'b--',                              #plots as a blue dashed line
                lw=1.5,                             #width1.5
                label=f'Plug-in h={h_x1_pi:.5f}')   #shows actual bandwidth rounded to 5dp

axes[0, 0].hist(sim_data['x1'], 
                bins=60,        #histogram with 60 intervals
                density=True,   #normalises (integrates to 1)
                alpha=0.2,      #reduces visibility
                color='blue')

axes[0, 0].set_title('Figure 1: KDE of $x_1$ — Plug-in Bandwidth')   #title
axes[0, 0].set_xlabel('$x_1$')      #x-axis label
axes[0, 0].set_ylabel('Density')    #y-axis label
axes[0, 0].legend()     #shows legend

# Figure 2: x2 Plug-in 
axes[0, 1].plot(x2_grid, f_x2_pi, 
                'r--', lw=1.5, 
                label=f'Plug-in h={h_x2_pi:.5f}')

axes[0, 1].hist(sim_data['x2'], 
                bins=60, density=True, 
                alpha=0.2, color='red')
axes[0, 1].set_title('Figure 2: KDE of $x_2$ — Plug-in Bandwidth')
axes[0, 1].set_xlabel('$x_2$')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend()

# Figure 3: x1 Plug-in & CV 
axes[1, 0].plot(x1_grid, f_x1_pi, 
                'b--', lw=1.5, 
                label=f'Plug-in h={h_x1_pi:.5f}') #plots plug-in KDE 
axes[1, 0].plot(x1_grid, f_x1_cv, 
                'b-', lw=2.0,                  #solid, thicker dashed line
                label=f'CV h={h_x1_cv:.5f}')    #plots CV KDE on the same axis

axes[1, 0].hist(sim_data['x1'], bins=60, density=True, 
                alpha=0.2, color='blue')    #plots histogram
axes[1, 0].set_title('Figure 3: KDE of $x_1$ — Plug-in vs CV Bandwidth')
axes[1, 0].set_xlabel('$x_1$')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend()

# Figure 4: x2 Plug-in & CV 
axes[1, 1].plot(x2_grid, f_x2_pi, 
                'r--', lw=1.5, 
                label=f'Plug-in h={h_x2_pi:.5f}')
axes[1, 1].plot(x2_grid, f_x2_cv, 
                'r-',  lw=2.0, 
                label=f'CV h={h_x2_cv:.5f}')

axes[1, 1].hist(sim_data['x2'], bins=60, 
                density=True, alpha=0.2, color='red')
axes[1, 1].set_title('Figure 4: KDE of $x_2$ — Plug-in vs CV Bandwidth')
axes[1, 1].set_xlabel('$x_2$')
axes[1, 1].set_ylabel('Density')
axes[1, 1].legend()

plt.tight_layout()  #adjusts spacing between subplots
plt.show()


Table 1 shows the choice of bandwidths for the plug-in rule being 0.09814 for $x_1$ and 0.05589 for $x_2$; LSCV chooses considerably smaller bandwidths of 0.03642 and 0.00418 respectively. LSCV bandwidth choices are about three times smaller for $x_1$ and nine times smaller for $x_2$.

For $x_1 ~ U(0.5, 2.5)$, Figures 1 and 3 show the plug-in KDE produces a smooth curve that captures the broadly uniform shape, but smooths out the local undulations visible in the histogram. The LSCV estimate, with its smaller bandwidth, tracks these fluctuations  more closely, producing a wigglier fit, with several local peaks and troughs. Since the plug-in's normality assumption is not severely violated by a near-uniform distribution, its over-smoothing is modest and arguably preferable: as the local bumps are sampling noise rather than genuine density features.

For $x_2 ~ Beta(0.5, 0.7)$, the difference is more noticeable. Figure 2 shows the plug-in KDE almost entirely misses the sharp spikes near 0 and 1 that characterise the U-shaped distribution, producing a near-flat estimate. Figure 4 shows the LSCV estimates more accurately at the boundaries. This is driven by the smaller bandwidth choice that uses only very close observations to predict the density, hence avoiding boundary bias. LSCV chooses this smaller bandwidth because it chooses the bandwidth empirically, so it finds the high curvature near the boundaries, in comparison  to the plug-in that imposes a normality assumption, underestimating the curvature.

In short, cross-validation outperforms the plug-in when the true density is far from normal, but risks overfitting noise in smooth regions. The difference in bandwidth choice between $x_1$ and $x_2$ illustrates this trade-off.

While the plots suggest CV outperforms the plug-in for $x_2$, it is difficult to tell how much worse the plug-in performs. To quantify this the Integrated Squared Error (ISE) can be compared to the true densities for both variables. Furthermore, to see whether the difference is concentrated among particular regions, a zoomed plot of the $x_2$ boundaries can be used to find where the plug-in performs most poorly.

In [ ]:
# Table 1 ---------------------
print("\nTable 1: Bandwidth Selection Results")
print("=" * 50)
print(bw_table.to_string())
print("=" * 50)

# ISE --------------------------
# Plug-in vs CV for x_1 against true U(0.5, 2.5) density 
def true_density_x1(x):
    return np.where((x >= 0.5) & (x <= 2.5), 0.5, 0.0)  #defines true density of x1 as 
                                                        #0.5 between 0.5-2.5 and 0 elsewhere

x1_ise_grid = np.linspace(0.5, 2.5, 1000)    #creates 1000 evaluation points between 0.2-2.5

f_x1_pi_ise = kde_epa(x1_ise_grid, sim_data['x1'].values, h_x1_pi)  #recompute KDE on the ISE grid (higher accuracy needed than for plotting)
f_x1_cv_ise = kde_epa(x1_ise_grid, sim_data['x1'].values, h_x1_cv)


f_true_ise = true_density_x1(x1_ise_grid)   #finds true U(0.5, 2.5) desnsity at every point

ise_pi = np.trapezoid((f_x1_pi_ise - f_true_ise)**2, x1_ise_grid)   #ISE approximated using trapezoid rule
ise_cv = np.trapezoid((f_x1_cv_ise - f_true_ise)**2, x1_ise_grid)   #squares error


# Plug-in vs CV for x_2 against true Beta(0.5, 0.7) density 
x2_ise_grid = np.linspace(0.001, 0.999, 1000)    #avoids exactly 0 and 1 where density goes to infinity

f_x2_pi_ise = kde_epa(x2_ise_grid, sim_data['x2'].values, h_x2_pi)  #recomputes plug-in and CV KDE for x2 on this grid
f_x2_cv_ise = kde_epa(x2_ise_grid, sim_data['x2'].values, h_x2_cv)
f_x2_true = beta_dist.pdf(x2_ise_grid, a=0.5, b=0.7)    #true probability density function at each point

ise_x2_pi = np.trapezoid((f_x2_pi_ise - f_x2_true)**2, x2_ise_grid)     #calculates ISE
ise_x2_cv = np.trapezoid((f_x2_cv_ise - f_x2_true)**2, x2_ise_grid)

# Table 2 
summary = pd.DataFrame({'Variable': ['x1', 'x2'],
                        'ISE Plug-in': [ise_pi, ise_x2_pi],
                        'ISE CV': [ise_cv, ise_x2_cv],
                        }).set_index('Variable').round(6) 

print("\nTable 2: ISE of Bandwidth Selection")
print("=" * 50)
print(summary.to_string())
print("=" * 50)



# Boundary Plot ----------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))     #creates 1x2 figure (2 subplots)

# Figure 5: Boundary plot near 0 
x2_left = np.linspace(0.001, 0.10, 500)     #500 points near left boundary where Beta density spikes 
f_pi_left = kde_epa(x2_left, sim_data['x2'].values, h_x2_pi)    #calculates plug-in KDE on grid
f_cv_left = kde_epa(x2_left, sim_data['x2'].values, h_x2_cv)    #calculates CV KDE
f_true_left = beta_dist.pdf(x2_left, a=0.5, b=0.7)   #calculates true density

axes[0].plot(x2_left, f_true_left, 'k-', lw=2,   
             label='True Beta(0.5, 0.7)')         #plots true density
axes[0].plot(x2_left, f_pi_left, 'r--', lw=1.5, 
             label=f'Plug-in h={h_x2_pi:.5f}')    #plots plug-in
axes[0].plot(x2_left, f_cv_left, 'r-',  lw=2,   
             label=f'CV h={h_x2_cv:.5f}')         #plots CV 
axes[0].set_title('Figure 5: $x_2$ KDE — Left Boundary')
axes[0].set_xlabel('$x_2$')
axes[0].set_ylabel('Density')
axes[0].legend()

# Figure 6: Boundary plot near 1
x2_right = np.linspace(0.90, 0.999, 500)
f_pi_right = kde_epa(x2_right, sim_data['x2'].values, h_x2_pi)
f_cv_right = kde_epa(x2_right, sim_data['x2'].values, h_x2_cv)
f_true_right = beta_dist.pdf(x2_right, a=0.5, b=0.7)

axes[1].plot(x2_right, f_true_right, 'k-', lw=2, 
             label='True Beta(0.5, 0.7)')
axes[1].plot(x2_right, f_pi_right, 'r--', lw=1.5, 
             label=f'Plug-in h={h_x2_pi:.5f}')
axes[1].plot(x2_right, f_cv_right, 'r-',  lw=2, 
             label=f'CV h={h_x2_cv:.5f}')
axes[1].set_title('Figure 6: $x_2$ KDE — Right Boundary')
axes[1].set_xlabel('$x_2$')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

The ISE results quantify what Figures 1-4 show. For $x_1$, the plug-in and CV perform similarly, with an ISE of 0.002806 and 0.002510 respectively. This small difference suggests the plug-in's larger bandwidth induces boundary bias, rather than an inability to capture the underlying distribution. For a uniform distribution where $f''(x)=0$ in the interior, bandwidth size has negligible effect on bias. Therefore, the three times bandwidth difference barely matters in practice.

For $x_2$, the difference is much more noticeable; the plug-in ISE of 0.210 is 6.3 times larger than the CV ISE of 0.033. This shows the cost of oversmoothing a U-shaped distribution and can be explained by the KDE bias formula where bias is proportional to $h²·f''(x)$. For $Beta(0.5, 0.7)$ the second derivative is large near the boundaries where the density spikes, so even a small increase in bandwidth creates large bias here. This explains why a bandwidth nine times larger gives an ISE that is 6.3 times greater.

Figures 5 and 6 show this, at the left boundary, the plug-in KDE sits at around 2 while the true density spikes to around 12 — a clear failure to estimate the distributional shape. The CV estimate tracks the true density closely in both boundary regions, however in Figure 6 the CV looks to overfit, producing visible oscillations. 


Overall, the results are as predicted. For $x_1$, both bandwidth selection methods perform similarly — as expected, since a uniform distribution is close to normal and the plug-in's normality assumption is not severely violated. The small ISE difference confirms that bandwidth choice is largely irrelevant when the true density is smooth and flat. For $x_2$, the predictions are also as expected. The plug-in rule selects a bandwidth 9x larger than CV, oversmooths the U-shaped Beta(0.5, 0.7) distribution, and produces an ISE 6.3x larger than CV. Figures 5 and 6 show this explicitly; the plug-in's normality assumption is violated by a distribution whose density lies at the extremes rather than its centre. Therefore, when possible, the bandwidth criterion should be matched to the assumed complexity of the underlying distribution.

***
## Part 3 — Out-of-Sample Forecast Comparison: Three Parametric Models

This section runs a 200-replication Monte Carlo simulation ($n=500$ per replication) to compare the out-of-sample forecasting performance of three OLS specifications, each estimated on 400 training observations and evaluated on the remaining 100:

$$\text{(Model A):} \quad y = \beta_0 + \beta_1 x_{1i} + \beta_2 x_{2i} + v_{Ai}$$

$$\text{(Model B):} \quad y = \gamma_0 + \gamma_1 \ln(x_{1i}) + \gamma_2 \ln(x_{2i}) + v_{Bi}$$

$$\text{(Model C):} \quad y = \delta_0 + \delta_1 \ln(x_{1i}) + \delta_2 x_{2i} + v_{Ci}$$

Since the true DGP follows $y = 0.1 + 0.5x_1 - 0.5\ln(x_2) + u$, only Model B is correctly specified. Models A and C each misspecify at least one regressor, allowing a direct test of how functional-form misspecification degrades forecast accuracy.

***


To compare the out-of-sample forecasting performance of three parametric OLS models across 200 Monte Carlo replications of the DGP, each replication draws $n = 500$ observations, estimates each model on the first 400 (training set), and forecasts the remaining 100 (test set).

The three models estimated are:

$$\text{(Model A):} \quad y = \beta_0 + \beta_1 x_{1i} + \beta_2 x_{2i} + v_{Ai}$$

$$\text{(Model B):} \quad y = \gamma_0 + \gamma_1 \ln(x_{1i}) + \gamma_2 \ln(x_{2i}) + v_{Bi}$$

$$\text{(Model C):} \quad y = \delta_0 + \delta_1 \ln(x_{1i}) + \delta_2 x_{2i} + v_{Ci}$$


Prediction accuracy is assessed using Mean Squared Prediction Error (MSPE), Root MSPE (RMSPE), and Mean Absolute Error (MAE). 

MSPE $= \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$ minimises the sum of squared residuals. It penalises large errors quadratically, meaning it is sensitive to larger forecast errors. The theoretical irreducible error is 1.0 as the variance of the error term is 1.

RMSPE $=\sqrt{\text{MSPE}}$ expresses the forecast error on the same scale as $y$, making it more interpretable. The theoretical irreducible error is 1.0 as $\sqrt{1} =1$

MAE $=\frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$ treats all errors equally regardless of size, so a model's success is not determined by how extreme errors are weighted. The theoretical irreducible error is derived from the expected absolute value, $E[|u|]=\sqrt{2/\pi}$ which is approximately $0.7979$ (from $u \sim N(0,1)$).

The forecast error can be decomposed to represent the bias-variance trade-off:

$$\mathbb{E}\left[(y - \hat{m}(x))^2\right] = \left(\mu(x) - \mathbb{E}[\hat{m}(x,\theta)]\right)^2 + \mathbb{E}\left[\left(\hat{m}(x,\theta) - \mathbb{E}[\hat{m}(x,\theta)]\right)^2\right] + \text{Var}(u)$$

where the first term represents the bias squared, the second term represents the variance, and the third term is the irreducible error.

The standard deviation of MSPE across replications is also found to determine whether differences in average performance are consistent or driven by a small number of outlier replications.



**Expected Results**

 The true DGP has the same magnitude (0.5) for both $x_1$ and $x_2$, so misspecifying either term should have a similar effect. However $x_2~Beta(0.5,0.7)$,is a highly non-linear, U-shaped distribution concentrated near 0 and 1. Therefore the difference between $x_2$ and $\ln(x_2)$ is much larger, and misspecifying will introduce substantial nonlinear bias. On the other hand $x_1 \sim U(0.5, 2.5)$ is a range over which $\ln(x_1)$ is approximately linear in $x_1$, so this misspecification introduces only a small approximation error. Model B, that only misspecifies $x_1$, is therefore expected to produce forecasts close to the irreducible error and expected to outperform both Models A and C. Model A misspecified only one coefficient, whereas Model C misspecifies both coefficients and so should perform the worst. 

In [ ]:
np.random.seed(SEED)

N_SIM = 200     # 200 repititions
N_TOTAL = 500   # 500 observations
N_TRAIN = 400   # training set of 400 obs
N_TEST = 100    # test set of 100 obs

mspe_A, mspe_B, mspe_C = [], [], []     #empty list to store MSPE for each model
mae_A, mae_B, mae_C = [], [], []        #stores MAE for each model

# Simulation loop
for s in range(N_SIM):      #loops 200 times
    df = dgp(n=N_TOTAL)     #creaetes new dataset of 500 observations using DGP
    tr = df.iloc[:N_TRAIN]  #first 400 rows used as training set, last 100 rows used as the test set
    te = df.iloc[N_TRAIN:]  #.iloc selects by row number instead of index label
    y_tr = tr['y'].values   #finds y for training set
    y_te = te['y'].values   #finds y for test set

    # Model A
    Xa_tr = sm.add_constant(tr[['x1', 'x2']])         #builds X matrices for training and test set
    Xa_te = sm.add_constant(te[['x1', 'x2']])         #adds intercept column of 1's
    yhat_A = sm.OLS(y_tr, Xa_tr).fit().predict(Xa_te) #estimates OLS on the training data
                                                      #then predicts on the test data
    mspe_A.append(np.mean((y_te - yhat_A)**2))        #MSPE for Model A and appends to storage lists
    mae_A.append(np.mean(np.abs(y_te - yhat_A)))      #MAE for Model A

    # Model B
    Xb_tr = sm.add_constant(np.column_stack([np.log(tr['x1']), np.log(tr['x2'])]))  
    Xb_te = sm.add_constant(np.column_stack([np.log(te['x1']), np.log(te['x2'])]))
                                                       #builds x matrices using log-transformed x1 and x2
                                                       #column_stack, combines the log columns into two column array
    yhat_B = sm.OLS(y_tr, Xb_tr).fit().predict(Xb_te)  #estimates OLS on training set, then predicts on the test data
    mspe_B.append(np.mean((y_te - yhat_B)**2))         #stores MSPE for Model B in storage lists
    mae_B.append(np.mean(np.abs(y_te - yhat_B)))       #MAE for Model B

    # Model C
    Xc_tr = sm.add_constant(np.column_stack([np.log(tr['x1']), tr['x2']]))
    Xc_te = sm.add_constant(np.column_stack([np.log(te['x1']), te['x2']]))
    yhat_C = sm.OLS(y_tr, Xc_tr).fit().predict(Xc_te)
    mspe_C.append(np.mean((y_te - yhat_C)**2))
    mae_C.append(np.mean(np.abs(y_te - yhat_C)))

mspe_A, mspe_B, mspe_C = map(np.array, [mspe_A, mspe_B, mspe_C]) #map applies np.array to each list 
mae_A, mae_B, mae_C = map(np.array, [mae_A, mae_B, mae_C])

# RMSPE
rmspe = lambda m: np.sqrt(np.mean(m)) #defined as a lambda(one-line function)
                                      #takes MSPE across all 200 replications then square roots

In [ ]:
# Table 3
summary = pd.DataFrame({'Model': ['A (x1, x2)', 'B (ln x1, ln x2)', 'C (ln x1, x2)'],
    'MSPE': [np.mean(mspe_A), np.mean(mspe_B), np.mean(mspe_C)],
    'RMSPE': [rmspe(mspe_A),   rmspe(mspe_B),   rmspe(mspe_C)],
    'MAE': [np.mean(mae_A),  np.mean(mae_B),  np.mean(mae_C)],
    'SD(MSPE)':[np.std(mspe_A),  np.std(mspe_B),  np.std(mspe_C)]}).set_index('Model').round(4)

print(f"\nTable 3: Out-of-Sample Performance ({N_SIM} replications, n={N_TOTAL})")
print("=" * 60)
print(summary.to_string())
print("=" * 60)


# Plots
labs = ['A\n(x1, x2)', 'B\n(ln x1,\nln x2)', 'C\n(ln x1, x2)']  #defines labels with breaks \n
cols = ['cornflowerblue', 'xkcd:powder pink', '#24bca8']
means_mspe = [np.mean(mspe_A), np.mean(mspe_B), np.mean(mspe_C)]
means_mae = [np.mean(mae_A),  np.mean(mae_B),  np.mean(mae_C)]
stds_mspe = [np.std(mspe_A),  np.std(mspe_B),  np.std(mspe_C)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))    #creates 3 plots

# Figure 7: MSPE bar chart
axes[0].bar(labs, means_mspe, 
            yerr=stds_mspe,             #adds error bars showing +/-1 STD
            capsize=6, color=cols)      #capsize adds gaps to bars
axes[0].axhline(1.0, color='black', linestyle='--', lw=1.5,
                label='Irreducible Error = 1')     #draws dashed line at MSPE=1.0
axes[0].set_ylabel('Mean MSPE')
axes[0].set_title('Figure 7: Mean MSPE (±1 SD)')
axes[0].legend()

# Figure 8: MAE bar chart
axes[1].bar(labs, means_mae, color=cols)
axes[1].axhline(np.sqrt(2/np.pi),       #plots benchmark line at ~0.7979
                color='black', linestyle='--', lw=1.5,
                label=f'Irreducible Error = {np.sqrt(2/np.pi):.3f}')    
axes[1].set_ylabel('Mean MAE')
axes[1].set_title('Figure 8: Mean MAE')
axes[1].legend()

# Figure 9: MSPE boxplot
axes[2].boxplot([mspe_A, mspe_B, mspe_C],   #plots full distribution of MSPE across all 200 repetions as boxplots
                tick_labels=labs,           #sets labels on the x-axis
                patch_artist=True)          #fills box with colour
for patch, colour in zip(axes[2].patches, cols):
    patch.set_facecolor(colour)         #sets colour of each box plot 
for median in axes[2].lines[4::6]:      #sets median colour (every 6th line starting at index 4)
    median.set_color('black')

axes[2].axhline(1.0, color='black', linestyle='--', lw=1.5,
                label='Irreducible Error = 1')  #plots irreducible error as dashed line
axes[2].set_ylabel('MSPE')
axes[2].set_title('Figure 9: MSPE Distribution (200 Replications)')
axes[2].legend()

plt.tight_layout()
plt.show()

Model B is the clear winner across all three measures of forecast error. Its MSPE of 1.0278, RMSPE of 1.0138, and MAE of 0.8088 sit within a fraction of a percent of their respective irreducible errors, confirming that correctly specifying $ln(x_2)$ gets rid almost all reducible approximation error. Its irreducible error varies from just 0.019 to 0.0278.

Models A and C perform nearly identically across each measure — MSPE of 1.4045 and 1.4077, RMSPE of 1.1851 and 1.1865, MAE of 0.9225 and 0.9239. This near-indistinguishability confirms that the main source of forecast error in both models is the misspecification of $x_2$ in levels rather than logarithms. The additional misspecification of $x_1$ in logarithms in Model C, has little effect. This mirrors the results in Task 2, where bandwidth differences had the greatest impact on predicting $x_2$'s density, less so for $x_1$. $x_2 ~ Beta(0.5, 0.7)$ is a nonlinear, U-shaped distribution whereas the difference between $x_2$ and $ln(x_2)$ is large, and so misspecifying this relationship introduces substantial bias. $x_1 ~ U(0.5, 2.5)$ is a range where $ln(x_1)$ is approximately linear in $x_1$, so misspecifying this relationship contributes much less additional error. 

The standard deviation of the MSPE's supports these conclusions. Model B's standard deviation of 0.1549 is much smaller than A's and C's (both around 0.267), visible in Figure 9 where Model B's box sits almost entirely below both A and C's. The median MSPE for B sits almost on the irreducible error line and has much smaller interquartile range than A and C. Model B is therefore not just more accurate on average but more consistent across all 200 replications.

These results are as expected. Model B was predicted to produce forecasts close to the irreducible error, and its MSPE excess of just 2.78% confirms this. Models A and C were expected to perform similarly and substantially worse — the near-identical MSPE values of 1.4045 and 1.4077, roughly 40% above the irreducible error, confirm both predictions. The cost of omitting the $ln(x_2)$ transformation dominates everything else, regardless of how $x_1$ is specified.

***
## Part 4 — Extending to Nonparametric Forecasting Models

This section extends the comparison to two nonparametric estimators, evaluated against the parametric Models A–C from Part 3:

$$\text{(Model D):} \quad \text{Nadaraya-Watson kernel regression}$$

$$\text{(Model E):} \quad \text{Polynomial series regression}$$

Both the implementation choices (kernel/bandwidth, polynomial degree selection) and the evaluation criteria are designed from scratch below, with reasoning given for each choice.

***


To compare the parametric models from Task 3 against two nonparametric models, the same simulation design is used : 200 replications and $n = 500$, with 400/100 train/test split.

**Model D** is the Nadaraya-Watson kernel regression estimator, the solution to:

$$\min_m \sum_{i=1}^{n} K\left(\frac{X_i - x}{h}\right)(Y_i - m)^2$$

giving:

$$\hat{m}(x) = \frac{\sum_{i=1}^{n} K_h(x - x_i) \cdot y_i}{\sum_{i=1}^{n} K_h(x - x_i)}$$

Since $X = (x_1, x_2)$ is two-dimensional:

$$K(v_1, v_2) = K(v_1) \cdot K(v_2)$$

with separate bandwidths $h_1, h_2$ for each dimension. Two bandwidth criteria are compared; firstly the Silverman rule of thumb $h_{\text{ROT}} = 1.06\hat{\sigma}n^{-1/5}$ as a baseline, and then the leave-one-out cross-validation minimising:

$$\text{LSCV}(h) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{m}_{-i}(x_i))^2$$

where $\hat{m}_{-i}(x_i)$ is the Nadaraya-Watson estimate at $x_i$ with observation $i$ removed.
In Model D, the Epanechnikov kernel is used for the Nadaraya-Watson estimator (consistent with Task 2), as only observations within the bandwidth window contribute to each local average, making it computationally efficient. Furthermore it is the asymptotically optimal kernel in the MSE sense among all valid kernel functions. The Silverman rule-of-thumb (ROT) and leave-one-out cross-validation (LOOCV) are chosen as the bandwidths for Model D. The ROT bandwidth is included as a baseline as it is fast to compute and performs well when inputs are approximately normal. However, given that $x_{2} \sim \text{Beta}(0.5, 0.7)$ is extremely non-normal, the ROT bandwidth is expected to oversmooth, hence the choice of LOOCV. 

**Model E** is a polynomial series regression estimator, this approximates $\mu(x)$ using:

$$m_l(x) = \sum_{j=1}^{k_l} z_{lj}(x)\beta_{lj}$$

where $z_{lj}(x)$ are polynomial basis functions of $(x_1, x_2)$ up to degree $d$, estimated by OLS. The degree $d$ is chosen using 5-fold cross-validation over $d \in \{2, 3, 4, 5\}$. The lower bound of 2 is chosen since $d = 1$ reduces to a misspecified linear model already covered by Models A–C; the upper bound of 5 is chosen since a degree-$d$ polynomial in 2 regressors produces $(d+1)(d+2)/2$ basis functions — degree 5 already gives 21 terms, beyond which near-multicollinearity becomes a concern with $n = 400$. 5-fold CV is used rather than leave-one-out CV as it provides a good bias-variance tradeoff in the cross-validation procedure itself and is substantially faster to compute across 200 replications.



**Expected Results**

The nonparametric models D and E are flexible enough to estimate the true DGP, and so are expected to outperform the very misspecified parametric Models A and C. However, the Nadarya-Watson's convergence rate depends on it's bandwidth choice, and with only $n=400$ training observations in $d=2$ dimensions, it will suffer from the curse of dimensionality. Model D will converge at the slower rate of $O(n^{-2/3})$ compared to the parametric rate of $O(n^{-1})$; it needs more data to achieve the same precision as it searches for the local average at each point. Similarly Model E's convergence rate depends on how fast the complexity of the model grows relative to the sample size. It will converge at the rate of  $O(n^{-2p/2p+d})$ where $p$ is th smoothness of the true model and $d(=2)$ is the number of dimensions.

Model D's performance is expected to be sensitive to bandwidth choice given the results found about $x_2$'s distribution in Task 2. Model E's performance will depend heavily on the degree selected by cross-validation.

In [ ]:
np.random.seed(SEED)

N_SIM = 200
N_TOTAL = 500
N_TRAIN = 400
N_TEST = 100

# Model D: Nadaraya-Watson kernel Regression --------------------------------   
def rot_bw(x):
    return 1.06 * np.std(x, ddof=1) * len(x)**(-0.2)    #defines Silverman ROT bandwidth


def nw_predict(x1_tr, x2_tr, y_tr, x1_te, x2_te, h1, h2):   #defines N-W predicor
                                                            #takes training/test data & bandwidths
    W = (K_epa((x1_te[:, None] - x1_tr[None, :]) / h1)     
         *K_epa((x2_te[:, None] - x2_tr[None, :]) / h2))    #builds(n_test × n_train) weight matrix using 2D product kernel (extends 1D kernels)
                                                            #multiplies Epanechnikov kernel on x1 distances by kernel applied to x2 distances
                                                            #each entry W[i,j] is the joint kernel weight between test point i and training point j
    denom = W.sum(axis=1)       #for each test point, sums kernel weights across all training points (normalises ensuring weights sum to 1)
    return (W @ y_tr) / denom   #finds weighted average of training y-values for each test point    
                                #(W@y_tr) is a matrix-vector product, divided by /denom to normalise    
                                #gives the NW estimator


def lscv_regression(h1, h2, x1_data, x2_data, y_data):         #LOOCV score for a given bandwith pair
    W = (K_epa((x1_data[:, None] - x1_data[None, :]) / h1) *    
         K_epa((x2_data[:, None] - x2_data[None, :]) / h2))    #product kernel weight matrix computed on training data against itself
                                                               #gives a (n_train × n_train) matrix of all pairwise weights
    
    np.fill_diagonal(W, 0.0)           #zeros the diagonal to removes self-influence
    denom = W.sum(axis=1)              #row sums of the zeroed weight matrix (normalises each prediction)
    
    with np.errstate(invalid='ignore', divide='ignore'):
        loo = np.where(denom > 0, (W @ y_data) / denom, np.nan) #finds LOOCV predicitions
                                                                #np.where assisngs nan when denom=0 (neighbouring points within h)
    
    ok = ~np.isnan(loo)                                                     #filters out nan predictions
    return np.mean((y_data[ok] - loo[ok])**2) if ok.sum() > 10 else 1e10    #returns MSLOOCV error
                                                                            #if less than 10 predictions returns large penalty (1e10)
                                                                            #avoids that bandwidth

def cv_bw_regression(x1_tr, x2_tr, y_tr):       #defines grid search for CV bandwidth
    h1_rot = rot_bw(x1_tr)      #initial ROT bandwidth for x1
    h2_rot = rot_bw(x2_tr)
    grid = [0.25, 0.5, 0.75, 1, 1.25, 1.5, 1.75, 2.0]  #searches bandwidths from 0.25-2x the ROT bandwidth in steps of 0.25
                                        
    best_h1, best_h2, best_cv = h1_rot, h2_rot, np.inf  #best values, starts at ROT bandwidth and attempts to beat np.inf
    for s1, s2 in product(grid, grid):      #loop over all 25 combinations of scale factors (5x5 grid), evaluates every (h1,h2) pair in the grid
        h1 = s1 * h1_rot                   
        h2 = s2 * h2_rot
        cv = lscv_regression(h1, h2, x1_tr, x2_tr, y_tr)    #for each bandwidth pair finds CV score   
        if cv < best_cv:                                    #if it improves the best_cv is updated
            best_cv, best_h1, best_h2 = cv, h1, h2
    return best_h1, best_h2                                 #returns optimal bandwidth pair



# Model E: Polynomial series regression -------------------------------
def cv_poly_degree(x1_tr, x2_tr, y_tr,      #defines best polynomial degree
                   degrees=[2, 3, 4, 5],    #searches over degrees 2-5
                   n_folds=5):              #uses 5-fold CV
    n = len(y_tr)                        
    fold_size = n // n_folds            #finds fold size by dividing sample size equally
    best_deg, best_mse = 2, np.inf      #sets best degree as 2 and best MSE as infinity
    
    for deg in degrees: 
        fold_mses = []  #outer loop to collect fold MSE for each degree
        for f in range(n_folds):     #inner loop over the 5-folds
            val_idx = np.arange(f * fold_size, (f + 1) * fold_size)         #defines validation indicex for fold f
            tr_idx = np.concatenate([np.arange(0, f * fold_size),       
                                     np.arange((f + 1) * fold_size, n)])    #defines training index
                                                                            #np.arrange selects rows before and after validation set
            poly = PolynomialFeatures(degree=deg, include_bias=True)        #creates a polynomial feature transformer 
                                                                            #include_bias=TRUE adds intercept
            Xe_tr = poly.fit_transform(np.c_[x1_tr[tr_idx], x2_tr[tr_idx]]) #np.c stacks x1 and x2 into a two-column matrix
                                                                            #fit_transform fits polynomial expansion on the training data
            Xe_val = poly.transform(np.c_[x1_tr[val_idx], x2_tr[val_idx]])  #transform applies same expansion to validation without refitting 
            
            pred = LinearRegression(fit_intercept=False).fit(Xe_tr, y_tr[tr_idx]).predict(Xe_val)       
                                                                    #fits OLS on expanded training data, predicts on validation
                                                                    #fit_intercept=False as intercept already included           
            fold_mses.append(np.mean((y_tr[val_idx] - pred)**2))    #calculates and stores MSE for the fold 
        avg_mse = np.mean(fold_mses)    #averages MSE across all 5 folds for this degree
        if avg_mse < best_mse:      #if MSE beats the current best it updates
            best_mse, best_deg = avg_mse, deg
    return best_deg

In [ ]:
# Simulation loop --------------------------------------------------------
degree_choices = []
results = {m: {'mspe': [], 'mae': []}                 #dictionary of dictionaries
           for m in ['A','B','C','D','D_cv','E']}     #one entry per model, each with two empty lists for MSPE and MAE

for s in range(N_SIM):
    df = dgp(n=N_TOTAL)
    tr = df.iloc[:N_TRAIN]; te = df.iloc[N_TRAIN:]  #generates fresh data each iteration and splits into test/train sets
    y_tr = tr['y'].values; y_te = te['y'].values
    x1_tr = tr['x1'].values; x2_tr = tr['x2'].values
    x1_te = te['x1'].values; x2_te = te['x2'].values    #extracts x1 and x2 as seperate np.arrays as Models D and E need them individually

    # Model A
    Xa = sm.add_constant(np.c_[x1_tr, x2_tr])
    yhat_A = sm.OLS(y_tr, Xa).fit().predict(sm.add_constant(np.c_[x1_te, x2_te]))
                                                                  

    # Model B
    Xb = sm.add_constant(np.c_[np.log(x1_tr), np.log(x2_tr)])
    yhat_B = sm.OLS(y_tr, Xb).fit().predict(sm.add_constant(np.c_[np.log(x1_te), np.log(x2_te)]))           
                                                                  

    # Model C
    Xc = sm.add_constant(np.c_[np.log(x1_tr), x2_tr])
    yhat_C = sm.OLS(y_tr, Xc).fit().predict(sm.add_constant(np.c_[np.log(x1_te), x2_te]))
                                                                 

    # Model D - ROT bandwidth
    h1_rot, h2_rot = rot_bw(x1_tr), rot_bw(x2_tr)   #computes ROT optimal bandwidths 
    yhat_D = nw_predict(x1_tr, x2_tr, y_tr, x1_te, x2_te, h1_rot, h2_rot)   #predicts using optimal bandiwths

    # Model D - CV bandwidth 
    h1_cv, h2_cv = cv_bw_regression(x1_tr, x2_tr, y_tr)     #runs grid search CV to find optimal bandwidths
    yhat_D_cv = nw_predict(x1_tr, x2_tr, y_tr, x1_te, x2_te, h1_cv, h2_cv)     #predicts using optimal bandwidths


    # Model E - CV degree selection
    best_deg = cv_poly_degree(x1_tr, x2_tr, y_tr)     #runs 5-fold CV to choose best degree from [2,5]
    poly = PolynomialFeatures(degree=best_deg, include_bias=True)
    Xe_tr = poly.fit_transform(np.c_[x1_tr, x2_tr])
    Xe_te = poly.transform(np.c_[x1_te, x2_te])
    yhat_E = LinearRegression(fit_intercept=False).fit(Xe_tr, y_tr).predict(Xe_te)   #fits OLS and predicts test set

    degree_choices.append(best_deg)    #records which degree was chosen in each iteration

    # Store errors
    for label, yhat in [('A', yhat_A), ('B', yhat_B), ('C', yhat_C),
                        ('D', yhat_D), ('D_cv', yhat_D_cv), ('E', yhat_E)]:    #loops over all models, simultanously pairing each model with its predictions
        errs = y_te - yhat      #calculates raw prediciton errors, a vector of 100 errors (one test per obs)
        results[label]['mspe'].append(np.mean(errs**2))      #computes and adds MSPE from error vector, appends to results dictionary
        results[label]['mae'].append(np.mean(np.abs(errs)))  #computes and adds MAE from error vector, appends to results dictionary

In [ ]:
# Table 4: Out-of-Sample Performance
model_keys = ['A', 'B', 'C', 'D', 'D_cv', 'E']
summary = pd.DataFrame({
    'Model': ['A — OLS: x1, x2', 'B — OLS: ln(x1), ln(x2)', 'C — OLS: ln(x1), x2',
              'D — NW (ROT)', 'D_cv — NW (CV)', 'E — Poly series (CV degree)'],
    'MSPE': [np.nanmean(results[m]['mspe']) for m in model_keys],
    'SD': [np.nanstd(results[m]['mspe'])  for m in model_keys],
    'MAE': [np.nanmean(results[m]['mae'])  for m in model_keys]
}).set_index('Model').round(4)

print(f"\nTable 4: Out-of-Sample Performance ({N_SIM} replications, n={N_TOTAL})")
print("=" * 60)
print(summary.to_string())
print("=" * 60)

# Plots
labels  = ['A', 'B', 'C', 'D', 'D_cv', 'E']
cols = ['cornflowerblue', 'xkcd:powder pink', '#24bca8', '#ffff81', '#cea2fd', '#fe2f4a']
mspe_arrays = [np.array(results[m]['mspe']) for m in model_keys]    #MSPE arrays per model
mae_arrays = [np.array(results[m]['mae'])  for m in model_keys]
means_mspe = [np.nanmean(a) for a in mspe_arrays]                   #mean MSPE per model
means_mae = [np.nanmean(a) for a in mae_arrays]
stds_mspe = [np.nanstd(a)  for a in mspe_arrays]                    #SD of MSPE per model

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].bar(labels, means_mspe, yerr=stds_mspe, capsize=6,
            color=cols, edgecolor='black', linewidth=0.6)   #MSPE bar chart with error bars
axes[0].axhline(1.0, color='black', linestyle='--', lw=1.5,
                label='Irreducible Error = 1.0')            #irreducible error line
axes[0].set_ylabel('Mean MSPE')
axes[0].set_title('Figure 8: Mean MSPE (±1 SD)')
axes[0].legend(fontsize=9)

axes[1].bar(labels, means_mae, color=cols,
            edgecolor='black', linewidth=0.6)
axes[1].axhline(np.sqrt(2/np.pi), color='black', linestyle='--', lw=1.5,
                label=f'Irreducible Error = {np.sqrt(2/np.pi):.3f}')
axes[1].set_ylabel('Mean MAE')
axes[1].set_title('Figure 9: Mean MAE')
axes[1].legend(fontsize=9)

bp = axes[2].boxplot(mspe_arrays, tick_labels=labels, patch_artist=True,
                     medianprops=dict(color='black', lw=1.5))   #MSPE boxplot
for patch, colour in zip(bp['boxes'], cols):
    patch.set_facecolor(colour) #apply model colour
    patch.set_alpha(0.7)        #transparency
axes[2].axhline(1.0, color='black', linestyle='--', lw=1.5,
                label='Irreducible Error = 1.0')
axes[2].set_ylabel('MSPE')
axes[2].set_title('Figure 10: MSPE Distribution (200 Replications)')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

### Interpretation

Among the nonparametric models $D_{CV}$ outperforms $D_{ROT}$ while $E$ outperforms both, confirming that empirical tuning methods improve performance. Cross-validation bandwidth selection reduces Model D's MSPE from 1.3332 to 1.2692 and MAE from 0.9028 to 0.8829, emphasising the earlier finding (Part 2) that the rule-of-thumb bandwidth oversmooths $x_2$'s non-normal (Beta(0.5,0.7)) distribution. The rule-of-thumb bandwidth assumes approximately normal inputs and selects a bandwidth too large to resolve the nonlinearity in $x_2$, whereas leave-one-out-cross-validation minimises the prediction error and selects a tighter bandwidth to better capture the local structure. 

However even with LOOCV bandwidth selection, Model $D_{CV}$ remains considerably inferior to Model B. This gap is due to the curse of dimensionality, with $n=400$ training observations the effective number of observations contributing to each local average is only approximately $n²/³ ≈ 54$, which is sufficient to detect broad trends but insufficient to realise the logarithmic nonlinearity in $x_2$, particularly with the Beta(0.5,0.7) distribution. The MSPE gap between $D_{CV}$ (1.2685) and B (1.0278) of 0.2407 quantifies the finite-sample cost of nonparametric flexibility in two dimensions.

Model E performs best out of the nonparametric models with a MSPE of 1.1972 and MAE of 0.8621. By imposing smooth global polynomial structure across the full training sample, Model E has lower variance than the local Nadaraya-Watson estimator and enough flexibility to approximate the logarithmic relationships in the true DGP. Unlike the Nadaya-Watson estimator which averages over only around 54 local observations, Model E uses all 400 training observations globally to estimate its polynomial coefficients, so it converges at a rate closer to the parametric rate. The lower standard deviation of MSPE for E (0.2049) compared to $D_{CV}$ (0.2295) and $D_{ROT}$ (0.2398) reflects this greater stability across replications. Despite this, Model E remains inferior to B, the polynomial basis, however flexible, cannot replicate the $ln(x_2)$ transformation that B specifies explicitly.

Models A and C remain essentially identical at MSPEs of 1.4045 and 1.4077, confirming again that misspecifying $x_2$ in levels creates the greatest error regardless of how $x_1$ is specified. From Figures 8-10, both A and C perform much worse than all nonparametric specifications; except $D_{ROT}$ that is notably closer to A and C. 

Figure 10 shows these findings visually. Model B's MSPE box is on the irreducible error line with the median just above, with the smallest IQR and a single outlier around 1.4; B's worst out of sample forecast still outperform A and C's average performance. A and C's boxplots are wide and centred well above the irreducible line with multiple outliers reaching around 2.4, showing the large variance that comes from a badly misspecified model. Model E's box is noticeably smaller than both Nadaraya-Watson models, again confirming the stability advantage of the polynomial approach over fully local estimation.


The figures and results table give a clear ranking of the models: $B > E > D_{CV} > D_{ROT} > A ≈ C$. To verify this ranking is robust to sampling variation across the 200 replications, 95% confidence intervals for the mean MSPE are constructed using:
$$\text{Mean MSPE} \pm 1.96 \times \left( \frac{\sigma_{\text{MSPE}}}{\sqrt{200}} \right)$$
where $\sigma_{\text{MSPE}}$ is the standard deviation of the MSPE across the simulations.


In [ ]:
# Table 5: Mean MSPE with 95% Confidence Intervals
model_keys  = ['A', 'B', 'C', 'D', 'D_cv', 'E']
model_names = ['A — OLS: x1, x2', 'B — OLS: ln(x1), ln(x2)', 'C — OLS: ln(x1), x2',
               'D — NW (ROT bandwidth)', 'D_cv — NW (CV bandwidth)', 'E — Poly series (CV degree)']

means = [np.nanmean(results[m]['mspe']) for m in model_keys]     #mean MSPE across 200 replications per model
ses = [np.nanstd(results[m]['mspe']) / np.sqrt(N_SIM) for m in model_keys]  #standard error = SD / sqrt(n_sims)

summary5 = pd.DataFrame({
    'Model': model_names,
    'Mean MSPE': means,
    'SE': ses,
    '95% CI Lo': [m - 1.96*s for m, s in zip(means, ses)],  #lower bound: mean - 1.96*SE
    '95% CI Hi': [m + 1.96*s for m, s in zip(means, ses)]   #upper bound: mean + 1.96*SE
}).set_index('Model').round(4)

print(f"\nTable 5: Mean MSPE with 95% Confidence Intervals ({N_SIM} replications)")
print("=" * 60)
print(summary5.to_string())
print("=" * 60)

# Figure 11
cols = ['cornflowerblue', 'xkcd:powder pink', '#24bca8', '#ffff81', '#cea2fd', '#fe2f4a']
means = [np.nanmean(results[m]['mspe']) for m in model_keys]
ses = [np.nanstd(results[m]['mspe'])/np.sqrt(N_SIM) for m in model_keys]

plt.figure(figsize=(10,5))
plt.bar(labels, means, yerr=[1.96*s for s in ses], #yerr adds 95%CI error bars
        capsize=6,      #adds horizontal error bars
        color=cols, edgecolor='black')
plt.axhline(1.0, color='black', linestyle='--', lw=1.5, label='Irreducible Error = 1.0')
plt.title('Figure 11: Mean MSPE with 95% Confidence Intervals')
plt.ylabel('Mean MSPE')
plt.legend()
plt.tight_layout()
plt.show()

The 95% confidence intervals confirm that every pairwise ranking in the results table is statistically robust. Model B's interval (1.0063, 1.0492) sits entirely below all other models. Even the lower bound of E's interval (1.1688) lies well above the upper bound of B's interval (1.0492), confirming that E's inferiority to B is a genuine feature of the models rather than sampling variation.

The ranking $B > E > D_{CV} > D_{ROT} > A \approx C$ is confirmed across all pairs, though the $D_{CV}$ and $D_{ROT}$ intervals (1.2367, 1.3003) and (1.3000, 1.3665) are almost adjacent with little separation, suggesting the performance difference between the two bandwidth selection methods is modest. The intervals for Models A and C  — (1.3675, 1.4414) and (1.3707, 1.4447) — overlap almost entirely, confirming they are statistically indistinguishable as shown in Task 3.

The narrowing of confidence intervals from (A to C to D to E to B) shows the increase in model stability as the models are better specified and tuned. This is consistent with the standard deviation results in Table 4.

To further investigate Model E's performance, the cross-validated chosen polynomial degree and it's associated out-of-sample MSPE are recorded across all 200 replications to verify whether cross-validation is choosing the optimal degree.

In [ ]:
# MSPE by selected degree
degree_mspe = {}                                            #empty dictionary to store MSPE values grouped by degree
for deg, mspe in zip(degree_choices, results['E']['mspe']): #loops over each simulation pairing degree chosen with its MSPE
    if deg not in degree_mspe:
        degree_mspe[deg] = []       #creates new list for this degree if not seen before
    degree_mspe[deg].append(mspe)   #appends MSPE to the list for this degree


degree_summary = pd.DataFrame({
    'Degree': sorted(degree_mspe.keys()),                                            #unique degrees chosen, sorted ascending
    'Count': [len(degree_mspe[d]) for d in sorted(degree_mspe.keys())],              #number of simulations that chose each degree
    'Frequency': [len(degree_mspe[d])/N_SIM for d in sorted(degree_mspe.keys())],    #proportion of simulations choosing each degree
    'Mean MSPE': [np.mean(degree_mspe[d]) for d in sorted(degree_mspe.keys())],      #mean MSPE across simulations that chose this degree
    'SD MSPE':   [np.std(degree_mspe[d])  for d in sorted(degree_mspe.keys())]       #SD of MSPE across those simulations
}).set_index('Degree')

degree_summary['Frequency'] = degree_summary['Frequency'].map('{:.1%}'.format)   #formats frequency as percentage
degree_summary[['Mean MSPE', 'SD MSPE']] = degree_summary[['Mean MSPE', 'SD MSPE']].round(4)    #rounds to 4dp

print("\nTable 6: CV-Selected Degree: Frequency and Out-of-Sample MSPE")
print("=" * 52)
print(degree_summary.to_string())
print("=" * 52)

Table 6 shows that cross-validation most frequently selects degree 4 (46% of replications) followed by degree 5 (36%), yet degree 3 actually produces the lowest out-of-sample MSPE of 1.1815 — lower than both degree 4 (1.1900) and degree 5 (1.2108). This suggests that 5-fold cross-validation is overfitting by one degree — it chooses degree 4 most often when degree 3 would generalise better. This demonstrates that with $n=400$, cross-validation's error estimates are noisy such that it favours the excessive complexity over the more efficient degree 3.

Given the true DGP, a degree 3 polynomial can capture the logarithmic curve of $x_2$ with only 10 basis functions, and so degrees 4 and 5 introduce flexibility that fits stochastic noise. This explains the increase in MSPE form 1.1815 to 1.2108.

The largest standard deviation for degree 5 (0.2290) highlights the variance cost of excessive flexibility, as high-order terms become sensitive to random noise. Since Degree 2 was selected in only 0.5% of cases, cross-validation successfully identified and rejected under-fitted models. 

Models D and E are asymptotically consistent, meaning the estimation error should vanish as the sample size increases. To demonstrate this asymptotic convergence, the simulation is repeated for a larger sample size ($n=500,  1000, 2500, 5000, 10,000$). While the custom LSCV bandwidth is feasible for $n=400$, the complexity of large-sample cross-validation requires a more efficient choice, hence the use of scikit-learn.

In [ ]:
np.random.seed(SEED)

def run_asymptotic_study(n_list=[500, 1000, 2500, 5000, 10000], n_reps=25):
    all_B, all_D, all_E = [], [], []   #outer lists to store results across sample sizes

    for n in n_list:                        #loop over each sample size
        rep_B, rep_D, rep_E = [], [], []    #inner lists to store MSPE for each replication

        for rep in range(n_reps):
            df = dgp(n)             #create a dataset of size n from the true DGP
            n_train = int(n * 0.8)  #use 80% of observations for training
            tr = df.iloc[:n_train]  #training set (first 80% of rows)
            te = df.iloc[n_train:]  #test set (remaining 20% of rows)

            y_tr, y_te = tr['y'].values, te['y'].values        #y for train and test
            x1_tr, x2_tr = tr['x1'].values, tr['x2'].values    #x1 and x2 for training
            x1_te, x2_te = te['x1'].values, te['x2'].values    #x1 and x2 for test
            X_tr = np.c_[x1_tr, x2_tr]  #stacks x varibles into a two-column matrix for training
            X_te = np.c_[x1_te, x2_te]

            # Model B
            Xb_tr = sm.add_constant(np.c_[np.log(x1_tr), np.log(x2_tr)])    # build training matrix with intercept
            Xb_te = sm.add_constant(np.c_[np.log(x1_te), np.log(x2_te)])
            mspe_B = np.mean((y_te - sm.OLS(y_tr, Xb_tr).fit().predict(Xb_te))**2)  #fit OLS on train, predict on test, compute MSPE

            # Model D - custom CV bandwidth
            h1, h2 = cv_bw_regression(x1_tr, x2_tr, y_tr)    #find optimal bandwidths using grid-search LOOCV
            mspe_D = np.mean((y_te - nw_predict(x1_tr, x2_tr, y_tr, x1_te, x2_te, h1, h2))**2)  #predict using NW estimator, compute MSPE

            # Model E - custom CV degree selection
            best_deg = cv_poly_degree(x1_tr, x2_tr, y_tr)   #choose best polynomial degree using cross-validation
            poly = PolynomialFeatures(degree=best_deg, include_bias=True)   #create polynomial feature transformer up to chosen degree
            Xe_tr = poly.fit_transform(X_tr)     #fit polynomial expansion on training data and transform
            Xe_te = poly.transform(X_te)         #apply same expansion to test data (no refit)
            mspe_E = np.mean((y_te - LinearRegression(fit_intercept=False).fit(Xe_tr, y_tr).predict(Xe_te))**2)
                    #fit OLS (intercept already included), predict on test, compute MSPE

            rep_B.append(mspe_B)    #store this replication's MSPE 
            rep_D.append(mspe_D)
            rep_E.append(mspe_E)

        all_B.append(rep_B) #store all replications for this sample size
        all_D.append(rep_D)
        all_E.append(rep_E)

    means_B = [np.mean(r) for r in all_B]   #calculte mean MSPE across replications
    means_D = [np.mean(r) for r in all_D]
    means_E = [np.mean(r) for r in all_E]
    ses_B = [np.std(r) / np.sqrt(n_reps) for r in all_B]    #calculate standard error of MSPE across replications
    ses_D = [np.std(r) / np.sqrt(n_reps) for r in all_D]    
    ses_E = [np.std(r) / np.sqrt(n_reps) for r in all_E]

    model_names = ([f'B (n={n})' for n in n_list] +      #build row labels for summary table (one row /model/sample size)
                   [f'D (n={n})' for n in n_list] +
                   [f'E (n={n})' for n in n_list])
    means_all = means_B + means_D + means_E     #mean MSPEs across all models
    ses_all   = ses_B   + ses_D   + ses_E       #standard errors across all models

    asymp_summary = pd.DataFrame({      #summary DataFrame with mean MSPE, SE, and 95% confidence intervals
        'Model': model_names,
        'Mean MSPE': means_all,
        'SE': ses_all,
        '95% CI Lo': [m - 1.96*s for m, s in zip(means_all, ses_all)],
        '95% CI Hi': [m + 1.96*s for m, s in zip(means_all, ses_all)]
    }).set_index('Model').round(4)

    print(f"\nTable 7: Asymptotic MSPE with 95% Confidence Intervals ({n_reps} replications)")
    print("=" * 65)
    print(asymp_summary.to_string())    #print full table
    print("=" * 65)

    return pd.DataFrame({'n': n_list, 'B': means_B, 'D': means_D, 'E': means_E})
        #DataFrame of mean MSPEs per sample size to use in the plot

asymp_df = run_asymptotic_study() #run the simulation and store results for plotting

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(asymp_df['n'], asymp_df['B'], label='Model B (Parametric)', marker='o', linestyle='--')
plt.plot(asymp_df['n'], asymp_df['D'], label='Model D (Kernel)', marker='s')
plt.plot(asymp_df['n'], asymp_df['E'], label='Model E (Series)', marker='^')
plt.axhline(1.0, color='red', linestyle=':', label='Irreducible Error (Var=1)')
plt.xscale('log') #log scale on x-axis to spread out the sample sizes evenly
plt.xlabel('Sample Size (n) - Log Scale')
plt.ylabel('MSPE')
plt.title('Figure 12: Convergence of Nonparametric Models as n -> Infinity')
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.5)
plt.tight_layout()
plt.show()

Figure 12 shows the MSPE for Models B, D, and E as the sample size grows from $n=500$ to $n=10{,}000$, illustrating the asymptotic convergence of both nonparametric models toward the irreducible error of 1.0.

At smaller sample sizes ($n \le 2500$), Model E outperforms the kernel estimator. This is likely due to its global polynomial structure, which captures the smooth logarithmic curvature of the DGP more effectively than local averaging when data is sparse. However, convergence is not completely monotonic — there is a slight increase in MSPE at $n=5000$. This instability suggests that while the polynomial basis is flexible, it can only approximate the logarithmic transformation to a certain degree of precision, regardless of sample size.

While Model D starts with the highest MSPE (1.3599 at $n=500$), it has a much steeper and more stable convergence rate. As data density increases, the local averaging of the Nadaraya-Watson estimator becomes increasingly precise — by $n=10{,}000$, Model D achieves an MSPE of 1.0828, overtaking Model E (1.1032). This suggests that local estimators eventually become preferable to global series approximations as the sample size grows, consistent with Model D's faster asymptotic convergence once enough data resolves its higher dimensionality cost.

In contrast, Model B remains virtually flat across all sample sizes, with MSPE values consistently near 1.00. This is the expected result for a correctly specified parametric model: because its functional form matches the DGP, it achieves the parametric convergence rate ($O(n^{-1})$) and reaches the irreducible error limit almost immediately. Even at $n=10{,}000$, the nonparametric models have not yet fully converged to the efficiency of Model B, illustrating the lingering "cost" of nonparametric flexibility.


**Conclusion**

Overall, the results confirm the expected ranking of $B > E > D_{CV} > D_{ROT} > A ≈ C$, further verified by the non-overlapping confidence intervals at the 95% confidence intervals. Model B remains the best forecasting model despite being misspecified in $x_1$ — correctly capturing the $ln(x_2)$ relationship is sufficient to recover most of the predictable variation in $y$.

The nonparametric models improve with better tuning — cross-validation bandwidth choice optimises Model D, and allows Model E to achieve the best nonparametric performance. Nevertheless, with $n=400$ training observations in two dimensions, the curse of dimensionality prevents any nonparametric model from outperforming Model B. The MSPE difference of 0.1694 represents the significant finite-sample cost of using a series estimator over a near-correct parametric model.

While nonparamatetric models are asymptotically consistent their slower convergence rates for $d=2$ suggests the sample size to close the 0.1694 gap is large. This highlights the efficiency of an approximately correct parametric model compared to the nonparametric flexibility when the sample size is not large.